# relay_agents

> Phase 23 (`specs/multiagent/phase23/`, experiment C1b): the agents' side of the relay board. The rule sentences `BRIDGE_ROLE_RULE` (a dedicated bridge's), `EVIDENCE_RULE` and `WINDOW_RULE` and `relayRules`, which `clue_colony.colonyRules` appends after `bridgeRules`; the bridge's pair line marks values confirmed, unconfirmed and contested on a relay board (`bridge_agents.bridgeLines`, FR-B6); `RoleBridge`, the numpy dedicated bridge, applied by `relayMembers` through `colony_board.stage4Members`, after `bridgeMembers`. Without a relay key nothing changes.

In [ ]:
#| default_exp relay_agents

In [ ]:
#| hide
from nbdev.showdoc import *

## Rules

The sentences come after the squad, couple, mirror and bridge sentences, which stay byte-identical (FR-B4, P20-15). A dedicated bridge gets `BRIDGE_ROLE_RULE` right after `BRIDGE_RULE` (FR-B1); every agent of an evidence board gets `EVIDENCE_RULE` (FR-B2); every agent of a window board gets `WINDOW_RULE` (FR-B3), and a window board has no bridge sentence since it has no bridges.

In [ ]:
#| export
from functools import lru_cache

from explRL.bridge_agents import RuleBridge, _vals
from explRL.clue_agents import RuleAgent
from explRL.clue_colony import Q, leading, tally
from explRL.colony_board import InSquad
from explRL.relay_board import RelayBoardEnv

BRIDGE_ROLE_RULE = ("Your role: bridge. Do not open or answer subtasks. Each turn, relay one value from squad {s2} that leads on its "
                    "mirror, is confirmed there and is not tied with another value, and that your squad's board does not already show "
                    "as leading; if a value you relayed has lost the lead on the mirror, relay the new leader; if there is none, share one "
                    "clue you hold that a board is missing, shows wrong or shows unconfirmed; if there is none, wait.")
EVIDENCE_RULE = ("A value marked 'relayed' was posted in squad {s2}; its count is the number of agents there who posted it, and other "
                 "values they post for that clue appear beside it.")
WINDOW_RULE = ("Values that squad {s2} posts for your subtasks that need it appear on your board, marked 'relayed', with the number "
               "of agents there who posted them.")


def relayRules(env, i):
    "Phase 23's sentences for agent i, each after a blank line ('' on any other board, or without i)."
    b = getattr(env, 'world', None)
    if i is None or not isinstance(b, RelayBoardEnv):
        return ''
    s2, out = (i + 1) % b.nSquads, []
    if i in b.roleBridges:
        out.append(BRIDGE_ROLE_RULE.format(s2=s2))
    if b.evidence:
        out.append(EVIDENCE_RULE.format(s2=s2))
    if b.mirrorWindow:
        out.append(WINDOW_RULE.format(s2=s2))
    return ''.join('\n\n' + x for x in out)

## The numpy dedicated bridge

`RoleBridge` acts only on a dedicated bridge's slot (`board.roleBridges`, read at act time); anyone else acts as its base class. It never opens. Each turn it takes, on each open coupled subtask of its pair in board order, the mirror's leader; it relays it when the leader has at least `q` senders, no other value has as many, its squad's board does not show it leading, and it has not relayed that value on that subtask already (so a value its board can never show leading, say against two receiving-squad liars, does not hold it on one subtask). Otherwise it shares a clue it holds as a guard does (on a mirror first, `RuleBridge._mirrorShare`), since the deal gives a bridge clues like any slot; otherwise it waits (FR-B5, AC-6). With evidence, the board shows the mirror's tally, so a new mirror leader already leads downstream and needs no second relay; without it, the bridge relays the new leader, which then leads as the newest value with one sender.

In [ ]:
#| export
class RoleBridge(RuleAgent):
    "Mixin over RuleBridge's classes: on a dedicated bridge's slot, relay an untied confirmed mirror leader the board lacks, else wait."
    def choose(self, b, t):
        if self.id in getattr(b, 'roleBridges', ()):
            return (self._roleRelay(b) or self._shareable(b, None, t, sorted(b.open, key=lambda k: (b.tasks[k].expire, k)))
                    or ('wait', None))
        return super().choose(b, t)

    def _roleRelay(self, b):
        p, q = b.bridgeOf(self.id), getattr(self, 'q', Q)
        if p is None or self.channel is None:
            return None
        tal = tally(self.channel, b)
        for k in b.open:
            if b.coupled.get(k) != p:
                continue
            mt = b.mirrorTally(k).get(k, {}).get(1, [])
            if not mt or len(mt[0][1]) < q or len(mt) > 1 and len(mt[1][1]) == len(mt[0][1]):
                continue
            lead, r = leading(tal, k, 1), int(mt[0][0])
            if (lead is None or lead[0] != r) and not any(x[1] == self.id and x[2] == k and x[4] == r for x in b.relayLog):
                return ('share', (k, 1, r))
        return None


@lru_cache(None)
def _roled(cls):
    return type('Role' + cls.__name__, (RoleBridge, cls), {})


def markedVals(p, vs, marks=True):
    "FR-B6: a pair line's values in colonyLine's words (confirmed, unconfirmed; contested), or C1's plain line without marks."
    if not (marks and vs):
        return _vals(p, vs)
    return ' | '.join(f"x mod {p} = {r} ({len(s)} agent{'s' * (len(s) != 1)}, {'confirmed' if len(s) >= Q else 'unconfirmed'})"
                      for r, s, _ in vs) + (', contested' if len(vs) > 1 else '')


def relayMembers(world, members):
    """colony_board.stage4Members' hook, after bridgeMembers: with bridgeRole, every RuleBridge member (honest rule agents) gains RoleBridge (class Role<class>),
    InSquad wrappers kept. Without it, or for Claude members, nothing changes. Returns `members`."""
    if not world.get('bridgeRole'):
        return members
    for m in members:
        x = m.m if isinstance(m, InSquad) else m
        if isinstance(x, RuleBridge) and not isinstance(x, RoleBridge):
            x.__class__ = _roled(type(x))
    return members

## Tests

In [ ]:
from explRL.bridge_agents import BRIDGE_RULE, COUPLE_RULE, MIRROR_RULE, PAIR_HEAD, bridgeLines, bridgeRules
from explRL.bridge_board import BridgeBoardEnv
from explRL.clue_agents import ClueBoard
from explRL.clue_colony import GUARD_RULE, SQUAD_RULE, colonyLine
from explRL.clue_text import TextClueEnv
from explRL.colony_board import squadView
from explRL.colony_grid import WORLD18

kw, G = dict(WORLD18), range(16, 32)
mk = lambda **x: RelayBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, **dict(dict(bridges=1), **x), **kw)
text = lambda b: TextClueEnv(b, colony=True, guards=16, nLiar=16, squads=8)

def act(b, ch, i, a, t):
    n = len(b.shareLog)
    out = b.apply(i, a, t)
    if len(b.shareLog) > n:
        x = b.shareLog[-1]
        ch.post(i, x[2], (x[3], b.tasks[x[2]].task.primes[x[3]], x[4]), t)
    return out

In [ ]:
# test_relay_rules (AC-5, FR-A4, FR-B1 to FR-B4, FR-B6): a dedicated bridge's rules are C1's bridge rules, then BRIDGE_ROLE_RULE;
# every agent of an evidence board gets EVIDENCE_RULE and of a window board WINDOW_RULE (no bridge sentence there); earlier
# sentences are byte-identical and come first; C1's boards get no new sentence; the pair line marks values on relay boards only
c1 = text(BridgeBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, bridges=1, **kw))
for x, role, ev, win in ((dict(evidence=True, bridgeRole=True), True, True, False), (dict(bridgeRole=True), True, False, False),
                         (dict(evidence=True), False, True, False), (dict(mirrorWindow=True, bridges=0), False, False, True)):
    b = mk(**x)
    env = text(b)
    for i in range(16, 64):
        txt, s, s2 = env.rules(i), i % 8, (i + 1) % 8
        assert txt.startswith(c1.rules(i) if b.bridges[0] else c1.rules(i).replace('\n\n' + BRIDGE_RULE.format(s2=s2), ''))
        assert (BRIDGE_ROLE_RULE.format(s2=s2) in txt) == (role and i in b.bridgeSet)
        assert (EVIDENCE_RULE.format(s2=s2) in txt) == ev and (WINDOW_RULE.format(s2=s2) in txt) == win
        assert (BRIDGE_RULE.format(s2=s2) in txt) == (i in b.bridgeSet) and (GUARD_RULE in txt) == (16 <= i < 32)
        if role and i in b.bridgeSet:
            assert txt.index(BRIDGE_RULE.format(s2=s2)) < txt.index(BRIDGE_ROLE_RULE.format(s2=s2))
        for r in (SQUAD_RULE.format(S=8, n=16, M=130), COUPLE_RULE.format(s2=s2), MIRROR_RULE.format(s0=(s - 1) % 8)):
            assert r in txt and txt.index(r) < len(c1.rules(i))
assert all(relayRules(c1, i) == '' for i in range(64)) and relayRules(text(mk()), 40) == '' and relayRules(text(mk(evidence=True)), None) == ''
# FR-B6: the bridge's pair line marks confirmed, unconfirmed and contested on a relay board; C1's pair line is unchanged
for x, marks in ((dict(bridgeRole=True), True), (dict(evidence=True), True), (dict(), False)):
    b = mk(**x) if x else BridgeBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, bridges=1, **kw)
    ch = ClueBoard()
    k = next(k for k in b.open if k in b.coupled)
    s, s2 = b.coupled[k]
    (p1, r1), br = b.tasks[k].task.clue(1), b.bridges[s][0]
    liar = next(i for i in range(16) if i % 8 == s2)
    for t, i in enumerate(sorted(b.tasks[k].holders[1]), 1):
        act(b, ch, i, ('share', (k, 1, r1)), t)
    act(b, ch, liar, ('share', (k, 1, (r1 + 1) % p1)), 3)
    line = next(y for y in bridgeLines(squadView(b, br), br) if y.startswith(PAIR_HEAD.format(s2=s2)))
    seg = line.split(f'T{k}: ')[1].split(';')[0]
    if marks:
        assert seg == f'x mod {p1} = {r1} (2 agents, confirmed) | x mod {p1} = {(r1 + 1) % p1} (1 agent, unconfirmed), contested', seg
    else:
        assert seg == f'x mod {p1} = {r1} (2 agents), x mod {p1} = {(r1 + 1) % p1} (1 agent)', seg

In [ ]:
# test_role_bridge_numpy (AC-6, FR-B5): RoleBridge never opens; it relays only an untied confirmed mirror leader its board lacks,
# waits otherwise, and relays a new leader; relayMembers applies it only with bridgeRole, keeping InSquad and the base class;
# on a 64-agent numpy board its slots never open and every relay was a confirmed untied leader when it was made
import numpy as np
from explRL.bridge_agents import RuleBridge, bridgeMembers
from explRL.clue_runs import clueSetup, runClueNumpy
from explRL.colony_agents import ColonyAgent, memberClass
from explRL.colony_board import InSquad, stage4Members
from explRL.llm_runs import LLMConfig

b, ch = mk(bridgeRole=True), ClueBoard()
k = next(k for k in b.open if k in b.coupled)
s, s2 = b.coupled[k]
(p1, r1), br = b.tasks[k].task.clue(1), b.bridges[s][0]
h1, h2 = sorted(b.tasks[k].holders[1])
liars = [i for i in range(16) if i % 8 == s2]
W = dict(kw, colony=True, guards=16, squads=8, couple=0.25, bridges=1, bridgeRole=True)
m = stage4Members(W, [ColonyAgent(ch, np.random.default_rng(0))])[0]
m.m.id = br
assert isinstance(m, InSquad) and type(m.m).__name__ == 'RoleBridgeColonyAgent' and isinstance(m.m, (RuleBridge, ColonyAgent))
v = squadView(b, br)
first = m.m.choose(v, 1)                                             # nothing on the mirror: share a held clue or wait, never open
assert first[0] in ('share', 'wait') and (first[0] == 'wait' or first[1][0] != k)
held = lambda a: a[0] == 'wait' or a[1][0] != k                      # anything but a relay on k
act(b, ch, liars[0], ('share', (k, 1, (r1 + 1) % p1)), 1)
assert held(m.m.choose(v, 2))                                          # one sender: unconfirmed
act(b, ch, h1, ('share', (k, 1, r1)), 2)
act(b, ch, liars[1], ('share', (k, 1, (r1 + 1) % p1)), 2)
act(b, ch, h2, ('share', (k, 1, r1)), 3)
assert held(m.m.choose(v, 3))                                          # 2 against 2: tied
act(b, ch, h2, ('share', (k, 1, r1)), 4)                              # the same sender again: still tied
assert held(m.m.choose(v, 4))
b2, ch2 = mk(bridgeRole=True), ClueBoard()
m.m.channel = ch2
v2 = squadView(b2, br)
for t, i in ((1, h1), (1, h2)):
    act(b2, ch2, i, ('share', (k, 1, r1)), t)
a = m.m.choose(v2, 2)
assert a == ('share', (k, 1, r1))
act(b2, ch2, br, a, 2)
assert b2.relayLog and held(m.m.choose(v2, 3))                       # its board shows it leading now
# a new leader after the relay: without evidence, the bridge relays it (AC-6's re-relay)
b3, ch3 = mk(bridgeRole=True), ClueBoard()
m.m.channel = ch3
v3 = squadView(b3, br)
for t, i in ((1, liars[0]), (1, liars[1])):
    act(b3, ch3, i, ('share', (k, 1, (r1 + 1) % p1)), t)
act(b3, ch3, br, m.m.choose(v3, 2), 2)
for t, i in ((3, h1), (3, h2), (4, next(x for x in range(16, 64) if x % 8 == s2 and x not in (h1, h2)))):
    b3.mirrorLog.append((float(t), i, k, 1, r1, True, False))         # three true senders (one planted) take the lead
assert m.m.choose(v3, 5) == ('share', (k, 1, r1))
# B (review): two receiving-squad liars hold a value the bridge's relay can never beat; it relays once, then moves on
b4, ch4 = mk(bridgeRole=True), ClueBoard()
m.m.channel = ch4
v4 = squadView(b4, br)
for t, i in ((1, h1), (1, h2)):
    act(b4, ch4, i, ('share', (k, 1, r1)), t)
for i in [x for x in range(16) if x % 8 == s][:2]:
    act(b4, ch4, i, ('share', (k, 1, (r1 + 1) % p1)), 1)
act(b4, ch4, br, m.m.choose(v4, 2), 2)
assert [x[4] for x in b4.relayLog] == [r1] and held(m.m.choose(v4, 3))
# A (review): a dedicated bridge shares a cross clue it holds on a mirror of squad s - 1 (the deal gives bridges clues)
b5 = mk(bridgeRole=True)
assert any(b5.held(br, x) for x in b5.open), 'the deal gives the bridge clues'
m.m.channel = ClueBoard()
a5 = m.m.choose(squadView(b5, br), 1)                                 # nothing to relay yet: it shares a true clue it holds
assert a5[0] == 'share' and br in b5.tasks[a5[1][0]].holders[a5[1][1]], a5
assert a5[1][2] == b5.tasks[a5[1][0]].task.clue(a5[1][1])[1], a5
# relayMembers: only with bridgeRole; a non-bridge slot acts as its base class
m0 = stage4Members(dict(W, bridgeRole=False), [ColonyAgent(ch, np.random.default_rng(0))])[0]
assert type(m0.m).__name__ == 'BridgeColonyAgent'
other = next(i for i in range(32, 64) if i % 8 == s and i != br)      # an honest non-bridge slot acts as its base class
m.m.id, m.m.channel = other, ch
ref = bridgeMembers(W, [InSquad(ColonyAgent(ch, np.random.default_rng(0)))])[0].m
ref.id, m.m.rng = other, np.random.default_rng(5)
ref.rng = np.random.default_rng(5)
assert other not in b.roleBridges and m.m.choose(squadView(b, other), 5) == ref.choose(squadView(b, other), 5)
# the numpy board: 64 agents, bridgeFix's world, 40 turns
cfg = LLMConfig(task='clue', channel='blackboard', N=64, turns=40, nLiar=16, memory='notes', world=dict(W, evidence=True))
from explRL.colony_trace import replay
rp = replay(cfg, 1800)
bd = rp.board
assert isinstance(bd, RelayBoardEnv) and bd.evidence and bd.roleBridges == bd.bridgeSet
assert not any(r[0] in bd.roleBridges and r[3] == 'travel' for r in bd.ledger)          # never opens
assert any(m[1] in bd.roleBridges for m in bd.mirrorLog) or any(s[1] in bd.roleBridges and (s[2], s[4]) not in {(x[2], x[4]) for x in bd.relayLog} for s in bd.shareLog)   # and shares held clues
assert len({(x[1], x[2], x[4]) for x in bd.relayLog}) == len(bd.relayLog)                  # no repeated relay
assert bd.relayLog and all(r[1] in bd.roleBridges for r in bd.relayLog)
def leaderAt(t, i, k):
    "The mirror's sender counts per value just before agent i acted at t."
    c = {}
    for x in bd.mirrorLog:
        if x[2] == k and (x[0] < t or x[0] == t and x[1] < i):
            c.setdefault(x[4], set()).add(x[1])
    return sorted(((len(s), r) for r, s in c.items()), reverse=True)
for t, i, k, j, r, _ in bd.relayLog:
    c = leaderAt(t, i, k)
    assert c[0][1] == r and c[0][0] >= 2 and (len(c) == 1 or c[1][0] < c[0][0]), (t, i, k, r, c)
print(len(bd.relayLog), 'relays;', sum(not (r[4] == bd.tasks[r[2]].task.clue(1)[1]) for r in bd.relayLog), 'false')

In [ ]:
# test_relay_prompt_tokens (NFR-4, P23-9): P23-9's bar (1,560) is on the pilot's measured mean input tokens per bridge call.
# Offline the estimate (bridgePromptTokens: longest prompt, about 4 characters a token, fake client) is scaled from C1's
# bridge1, whose estimate was 1,453 and whose measured mean was 1,002 (largest board mean 1,038): a dedicated bridge with
# evidence must stay within the cost allowance (1.15x) and project under 1,560. The longest 3B bridge prompt in A1's world
# (16 agents; squads of 8 with a dedicated bridge and evidence, squads of 4 with evidence and solver bridges, P23-7) must fit
# 3,072 by the same estimate (the 3B tokenizer's count is Phase 20 T7's)
from explRL.bridge_agents import bridgePromptTokens
from explRL.rl19_suite import world19e
w64 = dict(kw, colony=True, guards=16, squads=8, couple=0.25, bridges=1)
c64 = lambda **x: LLMConfig(task='clue', channel='blackboard', N=64, turns=40, nLiar=16, memory='notes', world=dict(w64, **x))
t1, t64 = bridgePromptTokens(c64(), [1800]), bridgePromptTokens(c64(evidence=True, bridgeRole=True), [1800])
ratio = t64['bridge'] / t1['bridge']
assert ratio < 1.15 and 1038 * ratio < 1560, (t1, t64)
for S, x in ((8, dict(evidence=True, bridgeRole=True)), (4, dict(evidence=True))):
    w = dict(world19e(S, guards=4, couple=0.25, bridges=1), **x)
    c = LLMConfig(task='clue', channel='blackboard', N=16, turns=40, nLiar=4, memory='notes', world=w)
    t = bridgePromptTokens(c, [900, 901])
    assert max(t.values()) < 3072, (S, t)
    print(S, t)
print(64, t1, t64, round(ratio, 3))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()